In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

In [3]:
df = pd.read_excel("../data/final_cleaned_dataset.xlsx")
df.head()

,EatLess_OnWeightGain,EatLess_AtMealtime,RefuseFood_WeightConcern,Monitor_Food,Eat_SlimmingFoods,EatLess_AfterOvereating,EatLess_ToPreventWeightGain,AvoidSnacks_BetweenMealsToWatchWeight,AvoidEveningEating_ToWatchWeight,ConsiderWeight_WhenEating,...,Days_ExcludedFoodControlShapeOrWeight_No days,Days_FollowedRulesControlShapeOrWeight_13-15 days,Days_FollowedRulesControlShapeOrWeight_6-12 days,Days_FollowedRulesControlShapeOrWeight_Every day,Days_FollowedRulesControlShapeOrWeight_No days,Days_FearLosingControlOverEating_13-15 days,Days_FearLosingControlOverEating_6-12 days,Days_FearLosingControlOverEating_Every day,Days_FearLosingControlOverEating_No days,target
0,3,3,3,3,3,3,3,3,3,2,...,False,False,False,True,False,False,False,True,False,Probably yes
1,3,2,2,3,2,4,3,4,3,2,...,False,False,False,True,False,False,True,False,False,Probably yes
2,0,0,0,0,0,0,0,0,0,0,...,True,False,False,False,True,False,False,False,True,"Possibly, but I am not sure"
3,2,2,1,3,2,1,3,1,0,2,...,True,False,False,False,True,False,False,False,False,Probably not
4,0,0,0,3,2,0,0,0,2,2,...,True,False,False,False,True,False,False,False,True,"Possibly, but I am not sure"


In [4]:
X = df.drop(columns=["target"]).astype(int)
y = df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (550, 104)
y shape: (550,)


In [5]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", list(le.classes_))

Classes: ['Possibly, but I am not sure', 'Probably not', 'Probably yes', 'Yes, I believe I do']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (440, 104)
Test shape: (110, 104)


In [7]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ]),
    
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(class_weight="balanced"))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ),
    
    "XGBoost": XGBClassifier(
        objective="multi:softmax",
        num_class=len(le.classes_),
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        eval_metric="mlogloss"
    )
}

In [8]:
results = []

for name, model in models.items():
    print(f"\n===== {name} =====")
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1_macro": f1
    })
    
    print("Accuracy:", acc)
    print("F1 Macro:", f1)
    print(classification_report(y_test, y_pred, target_names=le.classes_))


===== Logistic Regression =====
Accuracy: 0.41818181818181815
F1 Macro: 0.37777410865531985
                             precision    recall  f1-score   support

Possibly, but I am not sure       0.53      0.53      0.53        49
               Probably not       0.40      0.33      0.36        24
               Probably yes       0.30      0.30      0.30        23
        Yes, I believe I do       0.28      0.36      0.31        14

                   accuracy                           0.42       110
                  macro avg       0.38      0.38      0.38       110
               weighted avg       0.42      0.42      0.42       110


===== SVM =====
Accuracy: 0.41818181818181815
F1 Macro: 0.38249533684441517
                             precision    recall  f1-score   support

Possibly, but I am not sure       0.57      0.51      0.54        49
               Probably not       0.36      0.38      0.37        24
               Probably yes       0.28      0.30      0.29        2

In [9]:
results_df = pd.DataFrame(results).sort_values(by="F1_macro", ascending=False)
results_df

,Model,Accuracy,F1_macro
3,XGBoost,0.481818,0.388695
1,SVM,0.418182,0.382495
0,Logistic Regression,0.418182,0.377774
2,Random Forest,0.490909,0.357622
